# DIC2：50 Startups（主要）與 Boston Housing
魏柏勝 / 7114056191

CRISP-DM 工作流程；兩題共用 `regression_workflow.py`。本 Notebook 的顯示結果已實際執行。完整重訓由最後的重現指令執行。

In [1]:
from pathlib import Path
import json
import pandas as pd
ROOT = Path.cwd()
if not (ROOT / 'output').exists():
    ROOT = ROOT.parent if (ROOT.parent / 'output').exists() else ROOT / 'DIC2'
results = {key: json.loads((ROOT / 'output' / f'{key}_results.json').read_text()) for key in ['startup', 'boston']}
print('Loaded real Python analysis artifacts:', ', '.join(results))

Loaded real Python analysis artifacts: startup, boston


## 1. 資料理解與前處理
先切分 80/20（seed=0），再由訓練資料決定中位數與類別。50 Startups 的 State 使用 California 作參考類別，編碼後共 5 特徵；Boston 共 13 特徵，目標 MEDV 為歷史千美元。

In [2]:
for key, r in results.items():
    print(key, 'rows=', r['n'], 'train=', r['train_n'], 'test=', r['test_n'], 'features=', r['encoded_p'])
    print('quality:', r['quality'])

startup rows= 50 train= 40 test= 10 features= 5
quality: {'missing_cells': 0, 'duplicates': 0}
boston rows= 506 train= 404 test= 102 features= 13
quality: {'missing_cells': 0, 'duplicates': 0}


## 2. 模型選擇與評估
五種方法產生子集，每個子集重估 OLS。方法與 k 依訓練 5-fold CV 的 RMSE 選定；test RMSE 僅用於評估。Lasso / Elastic Net 在此是排序方法，最終預測器都是 OLS。CV 選擇分數不是無偏泛化誤差。

In [3]:
for key, r in results.items():
    best = r['records'][r['selected_index']]
    print('\n', r['name'], r['selection'])
    print('CV RMSE:', best['cv_rmse'], 'Test RMSE:', best['test_rmse'], 'Test MAE:', best['test_mae'], 'Test R2:', best['test_r2'])
    print('Full OLS test RMSE:', r['baseline']['test_rmse'])
    print(pd.DataFrame(r['records'])[['method','k','cv_rmse','test_rmse','test_r2']].to_string(index=False))


 50 Startups {'method': 'Forward SFS', 'k': 3, 'features': ['R&D Spend', 'Marketing Spend', 'State=Florida']}
CV RMSE: 9852.394241319143 Test RMSE: 8376.45051977164 Test MAE: 7053.029502882828 Test R2: 0.9451361446943114
Full OLS test RMSE: 9137.990152462438
     method  k      cv_rmse   test_rmse  test_r2
      Lasso  1 10023.277615 8274.868018 0.946459
      Lasso  2  9883.423749 8198.797191 0.947439
      Lasso  3 10064.689978 8803.775790 0.939396
      Lasso  4 10017.928933 9068.541185 0.935696
      Lasso  5 10305.187138 9137.990152 0.934707
Forward SFS  1 10023.277615 8274.868018 0.946459
Forward SFS  2  9883.423749 8198.797191 0.947439
Forward SFS  3  9852.394241 8376.450520 0.945136
Forward SFS  4 10017.928933 9068.541185 0.935696
Forward SFS  5 10305.187138 9137.990152 0.934707
        RFE  1 10023.277615 8274.868018 0.946459
        RFE  2  9883.423749 8198.797191 0.947439
        RFE  3 10064.689978 8803.775790 0.939396
        RFE  4 10017.928933 9068.541185 0.935696
     

## 3. 選定模型的係數與測試殘差
95% 係數區間與 p 值來自訓練 OLS；因先作特徵選擇，視為探索性推論。殘差定義為實際值減預測值。

In [4]:
for key, r in results.items():
    best = r['records'][r['selected_index']]
    print('\n', r['name'])
    print(pd.DataFrame(best['coefficients']).to_string(index=False))
    print(pd.DataFrame(best['residuals']).head(5).to_string(index=False))


 50 Startups
           name     estimate          se         t            p       ci_low      ci_high      vif
      Intercept 46530.077211 3069.509262 15.158800 3.329478e-17 40304.823891 52755.330531      NaN
      R&D Spend     0.786381    0.047976 16.390985 2.826275e-18     0.689080     0.883682 2.150337
Marketing Spend     0.033796    0.018188  1.858095 7.134844e-02    -0.003092     0.070684 2.208026
  State=Florida -1079.468169 3579.777751 -0.301546 7.647331e-01 -8339.593951  6180.657612 1.052136
 row    actual     predicted      residual  absolute_error
  11 146121.95 133337.815559  12784.134441    12784.134441
  39  81229.06  68699.437880  12529.622120    12529.622120
   3 191050.39 179900.604258  11149.785742    11149.785742
  28 105008.31 115170.279579 -10161.969579    10161.969579
  12 144259.40 134136.931108  10122.468892    10122.468892

 Boston Housing
     name   estimate       se         t            p     ci_low   ci_high      vif
Intercept  38.163773 5.477225  6.9677

## 4. 第二題直接套用工作流程
下面顯示同一套流程的核心函式；`--dataset boston` 只更換資料來源與目標，保留資料準備、子集選擇、評估、匯出步驟。完整兩題可執行 `python project/regression_workflow.py --dataset all`。

In [5]:
import importlib.util, inspect
spec = importlib.util.spec_from_file_location('regression_workflow', ROOT / 'project' / 'regression_workflow.py')
workflow = importlib.util.module_from_spec(spec)
spec.loader.exec_module(workflow)
print(inspect.getsource(workflow.run_dataset))

def run_dataset(key):
    start = time.time()
    df, target, name, unit, source = load_dataset(key)
    ids = np.arange(len(df))
    train_ids, test_ids = train_test_split(ids, test_size=0.2, random_state=0)
    Xtrain, Xtest, columns, specs = prepare(df, target, train_ids, test_ids)
    ytrain, ytest = df[target].to_numpy(float)[train_ids], df[target].to_numpy(float)[test_ids]
    print(f'{name}: selecting subsets on {len(train_ids)} training rows...', flush=True)
    candidates, winner, tuning = choose_paths(Xtrain, ytrain, columns)
    # Freeze the choice before the holdout participates in fitting or reporting.
    frozen = {'method': winner['method'], 'k': winner['k'], 'features': [columns[i] for i in winner['subset']]}
    records = [fit_record(r, Xtrain, ytrain, Xtest, ytest, columns, test_ids) for r in candidates]
    selected_index = next(i for i, r in enumerate(records) if r['method'] == winner['method'] and r['k'] == winner['k'])
    baseline = fit_record({'method': 'Full OL

## 5. 加分展示與限制
透過 `python -m http.server 8765 --directory site` 開啟網站。下拉選單切換兩題；自訂 CSV / URL 會在瀏覽器重新分析。Browser JavaScript 與 Python 的亂數演算法及參數網格不同，結果不強求相同。Boston 資料具有年代、MEDV 上限與 B 欄位的爭議限制，僅作課堂練習。